# LungRay SOTA Multi-Task Model Training
## Fine-tuning tiếp nối baseline đa nhiệm NIH / TB / COVIDx

Notebook này nạp checkpoint tốt nhất từ Notebook 02, sau đó fine-tune theo hai giai đoạn. Chạy huấn luyện trên Kaggle GPU với dữ liệu thật và các split đã kiểm định. Không có chế độ mock/smoke cho training.

| Hạng mục | Phương pháp |
|:---|:---|
| Mạng | DenseNet-121 dùng chung, ba head độc lập |
| Tổn thất | ASL (NIH), BCE (TB), Focal (COVIDx) |
| Sampler | 60% NIH : 25% COVIDx : 15% TB |
| Fine-tune | 3 epoch train heads, sau đó mở backbone với LR phân tầng |
| Chọn checkpoint | AUC tổng hợp 0.40 / 0.30 / 0.30 trên validation |
| Calibration | Youden's J tính từ dự đoán validation; test độc lập không tham gia |


In [17]:
# Cell 1: Tự Động Định Danh Môi Trường & Kết Nối Mã Nguồn (Smart Environment Discovery)
import os
import sys
import glob

print("[*] Đang kiểm tra môi trường thực thi...", flush=True)

# Xác định thư mục gốc của repository LungRay
project_root = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Hỗ trợ tự động kết nối trên Kaggle nếu repo được clone vào /kaggle/working
if os.path.exists("/kaggle/working/LungRay"):
    if "/kaggle/working/LungRay" not in sys.path:
        sys.path.insert(0, "/kaggle/working/LungRay")
    project_root = "/kaggle/working/LungRay"

print(f"[XÁC NHẬN] Thư mục mã nguồn LungRay: {project_root}")

[*] Đang kiểm tra môi trường thực thi...
[XÁC NHẬN] Thư mục mã nguồn LungRay: /kaggle/working/LungRay


In [18]:
# Cell 1.1: Tìm / kết nối source LungRay (offline-first, Kaggle paths resilient)
import glob
import os
import stat
import subprocess
import sys
import zipfile

REPO_URL = "https://github.com/philipsgn/LungRay.git"
WORK_REPO = "/kaggle/working/LungRay"


def _repo_from_ingestion(path):
    """Returns the repository root for a src/data/ingestion.py path."""
    return os.path.abspath(os.path.join(os.path.dirname(path), "..", ".."))


def _is_lungray_repo(root):
    """Checks that root contains the source modules needed by this notebook."""
    return all(
        os.path.isfile(os.path.join(root, relative))
        for relative in (
            "src/data/ingestion.py",
            "src/models/multi_task_net.py",
            "src/models/trainer.py",
        )
    )


def _find_repo():
    """Finds a complete source checkout before attempting any network access."""
    direct_roots = [
        WORK_REPO,
        project_root if "project_root" in globals() else "",
        os.getcwd(),
    ]
    for root in direct_roots:
        if root and _is_lungray_repo(root):
            return os.path.abspath(root)

    patterns = [
        os.path.join(os.getcwd(), "**", "src", "data", "ingestion.py"),
        "/kaggle/working/**/src/data/ingestion.py",
        "/kaggle/input/**/src/data/ingestion.py",
    ]
    seen = set()
    for pattern in patterns:
        for match in glob.iglob(pattern, recursive=True):
            candidate = _repo_from_ingestion(match)
            if candidate not in seen and _is_lungray_repo(candidate):
                return candidate
            seen.add(candidate)
    return None


def _extract_repo_zip(zip_path, destination):
    """Safely extracts a repository ZIP while rejecting traversal and symlinks."""
    destination = os.path.realpath(destination)
    with zipfile.ZipFile(zip_path) as archive:
        members = archive.infolist()
        if not any(name.filename.replace("\\", "/").endswith("src/data/ingestion.py") for name in members):
            return False
        for member in members:
            normalized = member.filename.replace("\\", "/")
            target = os.path.realpath(os.path.join(destination, *normalized.split("/")))
            if os.path.commonpath((destination, target)) != destination:
                raise ValueError(f"Unsafe path in repository ZIP: {member.filename}")
            mode = member.external_attr >> 16
            if stat.S_ISLNK(mode):
                raise ValueError(f"Symlinks are not accepted in repository ZIP: {member.filename}")
        archive.extractall(destination)
    return True


repo_root = _find_repo()
if repo_root is None:
    print("[*] Chưa thấy source local; thử clone LungRay vào /kaggle/working...")
    try:
        if os.path.isdir(os.path.join(WORK_REPO, ".git")):
            subprocess.run(
                ["git", "-C", WORK_REPO, "pull", "--ff-only"],
                check=True, timeout=120, capture_output=True, text=True,
            )
        else:
            subprocess.run(
                ["git", "clone", "--depth", "1", REPO_URL, WORK_REPO],
                check=True, timeout=180, capture_output=True, text=True,
            )
    except Exception as exc:
        print(f"[!] Không lấy được source qua Git: {exc}")
    repo_root = _find_repo()

if repo_root is None and os.path.isdir("/kaggle/input"):
    print("[*] Git không khả dụng; tìm ZIP source trong Kaggle Input...")
    zip_candidates = sorted(glob.iglob("/kaggle/input/**/*.zip", recursive=True))
    for zip_path in zip_candidates:
        try:
            if _extract_repo_zip(zip_path, "/kaggle/working"):
                repo_root = _find_repo()
                if repo_root:
                    print(f"[OK] Đã lấy source từ ZIP: {zip_path}")
                    break
        except (OSError, ValueError, zipfile.BadZipFile) as exc:
            print(f"[!] Bỏ qua ZIP không hợp lệ {zip_path}: {exc}")

if repo_root is None:
    raise FileNotFoundError(
        "Không tìm thấy source LungRay. Bật Internet để clone GitHub hoặc Add Data > "
        "Upload dataset ZIP có thư mục src/data/ingestion.py, rồi chạy lại Cell 1.1."
    )

# Refresh only a Git checkout under /kaggle/working; if offline, keep the source found.
if repo_root.startswith("/kaggle/working") and os.path.isdir(os.path.join(repo_root, ".git")):
    try:
        subprocess.run(
            ["git", "-C", repo_root, "pull", "--ff-only"],
            check=True, timeout=120, capture_output=True, text=True,
        )
        print("[OK] Đã cập nhật source từ GitHub.")
    except Exception as exc:
        print(f"[!] Không pull được (offline hoặc checkout thay đổi); dùng source hiện có: {exc}")

repo_root = os.path.abspath(repo_root)
project_root = repo_root
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

# Kiểm tra import ngay tại cell này để không đợi đến Cell 3 mới phát hiện lỗi.
import importlib.util
if importlib.util.find_spec("src") is None:
    raise ModuleNotFoundError(f"Không import được package 'src' từ repository: {repo_root}")
print(f"[OK] Đã kết nối source LungRay: {repo_root}")


[OK] Đã cập nhật source từ GitHub.
[OK] Đã kết nối source LungRay: /kaggle/working/LungRay


In [20]:
# Cell 2: Khởi Tạo Thiết Bị Phần Cứng & Cố Định Hạt Giống Ngẫu Nhiên (Reproducibility)
import random
import numpy as np
import torch

def configure_hardware_and_seeds(seed: int = 42) -> torch.device:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        device = torch.device("cuda")
        print(f"[PHẦN CỨNG] Kích hoạt GPU: {torch.cuda.get_device_name(0)}")
        print(f"[BỘ NHỚ VRAM] Khả dụng: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    else:
        device = torch.device("cpu")
        print("[PHẦN CỨNG] Chạy chế độ CPU Fallback (Khuyến nghị dùng Kaggle GPU P100/T4 để huấn luyện trọn vẹn 25 epochs)")
    return device

device = configure_hardware_and_seeds(42)

[PHẦN CỨNG] Kích hoạt GPU: Tesla T4
[BỘ NHỚ VRAM] Khả dụng: 14.56 GB


In [ ]:
# Cell 3: Tự Động Phân Giải Đường Dẫn Dữ Liệu & Kiểm Tra 8 Ràng Buộc Sống Còn
from src.data.ingestion import resolve_dataset_paths

dataset_paths = resolve_dataset_paths()
if os.path.isdir("/kaggle/input"):
    kaggle_dataset_overrides = {
        "nih_dir": "/kaggle/input/datasets/phanphat/lungray-nih-512/nih_512",
        "covid_dir": "/kaggle/input/datasets/phanphat/lungray-covidx-512/covidx_512",
        "tb_dir": "/kaggle/input/datasets/tawsifurrahman/tuberculosis-tb-chest-xray-dataset/TB_Chest_Radiography_Database",
    }
    for dataset_key, dataset_path in kaggle_dataset_overrides.items():
        if not os.path.isdir(dataset_path):
            raise FileNotFoundError(f"Kaggle dataset mount không tồn tại: {dataset_path}")
        dataset_paths[dataset_key] = dataset_path
print("[*] Danh mục đường dẫn các bộ dữ liệu lồng ngực:")
print(f" - Dữ liệu NIH ChestX-ray14 : {dataset_paths['nih_dir']}")
print(f" - Dữ liệu COVIDx CXR-4     : {dataset_paths['covid_dir']}")
print(f" - Dữ liệu Tuberculosis (TB): {dataset_paths['tb_dir']}")
print(f" - Thư mục Splits chỉ mục   : {dataset_paths['split_dir']}")
print(f" - Thư mục Lưu Trọng Số     : {dataset_paths['output_dir']}")

os.makedirs(dataset_paths['output_dir'], exist_ok=True)
os.makedirs(dataset_paths['results_dir'], exist_ok=True)

In [ ]:
# Cell 4: Tạo loaders từ split thật, kiểm tra leakage / source strata và đường dẫn ảnh
import os
import pandas as pd
from torch.utils.data import DataLoader, ConcatDataset
from src.data.dataset import ChestXrayDataset
from src.data.ingestion import load_covid_dataframe
from src.data.sampler import WeightedMultiTaskBatchSampler
from src.data.transforms import get_train_transforms, get_val_transforms
from src.models.trainer import multitask_collate_fn

assert device.type == "cuda", "Bật Kaggle GPU trước khi chạy training notebook 03."

# Resolve the split paths here from Cell 3's return value. Do not rely on
# notebook globals that may not have been created in a fresh Kaggle session.
nih_split_file = dataset_paths.get("nih_split_csv")
covid_split_file = dataset_paths.get("covid_split_csv")
tb_split_file = dataset_paths.get("tb_split_csv")
split_files = {
    "NIH patient_splits.csv": nih_split_file,
    "TB tb_splits.csv": tb_split_file,
}
for split_name, split_path in split_files.items():
    if not split_path:
        raise FileNotFoundError(
            f"Không tìm thấy {split_name}; kiểm tra thư mục splits được resolve: "
            f"{dataset_paths.get('split_dir')}"
        )
    if not os.path.isfile(split_path):
        raise FileNotFoundError(f"{split_name} không tồn tại tại: {split_path}")

df_nih = pd.read_csv(nih_split_file)
df_tb = pd.read_csv(tb_split_file)

# COVIDx manifests define the selected cohort, filenames, labels, patient IDs,
# and train/test partitions. Never join the unrelated Radiography split by order.
covid_manifest = load_covid_dataframe(
    dataset_paths["covid_dir"],
    split_dirs=[dataset_paths["split_dir"], os.path.join(project_root, "dataset", "splits")],
    val_ratio=0.10, seed=42,
)
required_manifest_cols = {"filename", "patient_id", "split", "label"}
missing_manifest_cols = required_manifest_cols - set(covid_manifest.columns)
if missing_manifest_cols:
    raise RuntimeError(f"COVIDx manifests thiếu cột bắt buộc: {sorted(missing_manifest_cols)}")

def _image_key(value: object) -> str:
    """Normalizes a source metadata image reference to a manifest filename."""
    return os.path.basename(str(value).replace("\\", "/")).strip()

manifest = covid_manifest.copy()
manifest["_image_key"] = manifest["filename"].map(_image_key)
manifest_keys = set(manifest["_image_key"])
if not manifest_keys or not manifest["_image_key"].is_unique:
    raise RuntimeError("COVIDx manifest filenames are empty or duplicated after normalization.")

# Search split/source sidecars in the selected dataset and Kaggle inputs. The
# legacy repo CSV is only accepted if every row matches the COVIDx manifest.
source_metadata_files = []
if covid_split_file and os.path.isfile(covid_split_file):
    source_metadata_files.append(covid_split_file)
source_metadata_files.extend(
    glob.glob(os.path.join(dataset_paths["covid_dir"], "**", "*.csv"), recursive=True)
)
if os.path.isdir("/kaggle/input"):
    source_metadata_files.extend(glob.glob("/kaggle/input/**/covid_splits.csv", recursive=True))
source_metadata_files = list(dict.fromkeys(path for path in source_metadata_files if os.path.isfile(path)))
source_diagnostics = []
df_covid = None
source_col = None

for metadata_path in source_metadata_files:
    try:
        header = pd.read_csv(metadata_path, nrows=0).columns.tolist()
    except Exception as exc:
        source_diagnostics.append(f"{metadata_path}: cannot read header ({exc})")
        continue
    image_col = next((col for col in ("filename", "Image Index", "image_id") if col in header), None)
    candidate_source_col = next((col for col in ("data_source", "source") if col in header), None)
    if image_col is None or candidate_source_col is None or "split" not in header:
        continue
    patient_col_candidate = next(
        (col for col in ("patient_id", "Patient ID", "patientId") if col in header), None
    )
    usecols = [image_col, candidate_source_col, "split"]
    if patient_col_candidate:
        usecols.append(patient_col_candidate)
    metadata_df = pd.read_csv(metadata_path, usecols=usecols)
    metadata_df["_image_key"] = metadata_df[image_col].map(_image_key)
    if not metadata_df["_image_key"].is_unique:
        source_diagnostics.append(f"{metadata_path}: duplicate filenames")
        continue
    matched_keys = manifest_keys.intersection(metadata_df["_image_key"])
    if matched_keys != manifest_keys:
        source_diagnostics.append(
            f"{metadata_path}: matched {len(matched_keys):,}/{len(manifest_keys):,} COVIDx filenames"
        )
        continue

    source_map = metadata_df[["_image_key", candidate_source_col, "split"]].rename(
        columns={candidate_source_col: "data_source", "split": "source_split"}
    )
    if patient_col_candidate:
        source_map["source_patient_id"] = metadata_df[patient_col_candidate].astype(str)
    joined = manifest.merge(source_map, on="_image_key", how="left", validate="one_to_one")
    if joined["data_source"].isna().any():
        source_diagnostics.append(f"{metadata_path}: missing source labels")
        continue
    split_matches = joined["split"].astype(str).str.lower() == joined["source_split"].astype(str).str.lower()
    if not split_matches.all():
        source_diagnostics.append(f"{metadata_path}: split assignments differ from COVIDx manifests")
        continue
    if patient_col_candidate and not (
        joined["patient_id"].astype(str) == joined["source_patient_id"].astype(str)
    ).all():
        source_diagnostics.append(f"{metadata_path}: patient IDs differ from COVIDx manifests")
        continue
    df_covid = joined.drop(columns=["_image_key", "source_split", "source_patient_id"], errors="ignore")
    source_col = "data_source"
    print(f"[COVIDx] Source metadata matched all manifest rows: {metadata_path}")
    break

if df_covid is None:
    raise RuntimeError(
        "COVIDx manifests đã nạp nhưng chưa tìm thấy source metadata khớp toàn bộ filename, "
        "patient_id và split. covid_splits.csv trong repo là cohort COVID-19 Radiography "
        "(khoảng 21k ảnh), không phải COVIDx CXR-4 (76,345 ảnh), nên không được ghép. "
        "Hãy Add Data một CSV đúng cohort có filename, data_source, patient_id, split. "
        "Các file đã kiểm tra: " + repr(source_diagnostics)
    )

required_splits = {"train", "val", "test"}
for name, frame in (("NIH", df_nih), ("COVIDx", df_covid), ("TB", df_tb)):
    if not required_splits.issubset(set(frame["split"].astype(str))):
        raise RuntimeError(f"{name}: thiếu split train/val/test")

# NIH split bắt buộc theo bệnh nhân, tuyệt đối không chia theo ảnh.
patient_col = next((c for c in ("Patient ID", "Patient_ID") if c in df_nih.columns), None)
assert patient_col is not None, "NIH split phải có Patient ID để kiểm tra leakage."
patient_sets = {s: set(df_nih.loc[df_nih["split"] == s, patient_col].dropna()) for s in required_splits}
assert patient_sets["train"].isdisjoint(patient_sets["val"])
assert patient_sets["train"].isdisjoint(patient_sets["test"])
assert patient_sets["val"].isdisjoint(patient_sets["test"])

# COVIDx cần giữ thông tin nguồn ảnh và patient id để kiểm soát shortcut/leakage.
assert source_col == "data_source", "COVIDx source metadata was not validated against manifests."
covid_patient_col = next((c for c in ("patient_id", "Patient ID", "patientId") if c in df_covid.columns), None)
if covid_patient_col:
    covid_patient_sets = {s: set(df_covid.loc[df_covid["split"] == s, covid_patient_col].dropna()) for s in required_splits}
    assert covid_patient_sets["train"].isdisjoint(covid_patient_sets["val"])
    assert covid_patient_sets["train"].isdisjoint(covid_patient_sets["test"])
    assert covid_patient_sets["val"].isdisjoint(covid_patient_sets["test"])
print("[PASS] NIH patient split; COVIDx manifest patient IDs and source-aligned splits verified.")

train_nih_df = df_nih[df_nih["split"] == "train"].reset_index(drop=True)
val_nih_df = df_nih[df_nih["split"] == "val"].reset_index(drop=True)
train_covid_df = df_covid[df_covid["split"] == "train"].reset_index(drop=True)
val_covid_df = df_covid[df_covid["split"] == "val"].reset_index(drop=True)
train_tb_df = df_tb[df_tb["split"] == "train"].reset_index(drop=True)
val_tb_df = df_tb[df_tb["split"] == "val"].reset_index(drop=True)
col_covid = next(c for c in ("filename", "Image Index", "image_id") if c in df_covid.columns)
col_tb = next(c for c in ("filename", "Image Index", "image_id") if c in df_tb.columns)

train_transform = get_train_transforms(img_size=512)
val_transform = get_val_transforms(img_size=512)
BATCH_SIZE, NUM_WORKERS = 16, 4

def make_dataset(frame, image_dir, image_col, label_col, task_id, num_classes, transform, pre_resized):
    """Builds a strict dataset; missing images raise instead of becoming mock samples."""
    return ChestXrayDataset(
        df=frame, image_dir=image_dir, image_col=image_col, label_col=label_col,
        task_id=task_id, num_classes=num_classes, transforms=transform,
        pre_resized=pre_resized, ignore_missing=False,
    )

ds_train_nih = make_dataset(train_nih_df, dataset_paths["nih_dir"], "Image Index", "Finding Labels", 0, 14, train_transform, dataset_paths["nih_pre_resized"])
ds_train_covid = make_dataset(train_covid_df, dataset_paths["covid_dir"], col_covid, "label", 2, 1, train_transform, dataset_paths["covid_pre_resized"])
ds_train_tb = make_dataset(train_tb_df, dataset_paths["tb_dir"], col_tb, "label", 1, 1, train_transform, True)
assert min(len(ds_train_nih), len(ds_train_covid), len(ds_train_tb)) > 0, "Một nguồn train rỗng."
concat_train = ConcatDataset([ds_train_nih, ds_train_covid, ds_train_tb])
offset_covid = len(ds_train_nih)
offset_tb = offset_covid + len(ds_train_covid)
batch_sampler = WeightedMultiTaskBatchSampler(
    nih_indices=range(0, offset_covid), covid_indices=range(offset_covid, offset_tb),
    tb_indices=range(offset_tb, len(concat_train)), batch_size=BATCH_SIZE,
    ratios=(0.60, 0.25, 0.15), shuffle=True, seed=42,
)
train_loader = DataLoader(
    concat_train, batch_sampler=batch_sampler, collate_fn=multitask_collate_fn,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0,
)

ds_val_nih = make_dataset(val_nih_df, dataset_paths["nih_dir"], "Image Index", "Finding Labels", 0, 14, val_transform, dataset_paths["nih_pre_resized"])
ds_val_covid = make_dataset(val_covid_df, dataset_paths["covid_dir"], col_covid, "label", 2, 1, val_transform, dataset_paths["covid_pre_resized"])
ds_val_tb = make_dataset(val_tb_df, dataset_paths["tb_dir"], col_tb, "label", 1, 1, val_transform, True)
assert min(len(ds_val_nih), len(ds_val_covid), len(ds_val_tb)) > 0, "Một validation dataset rỗng."
loader_args = dict(batch_size=BATCH_SIZE, shuffle=False, collate_fn=multitask_collate_fn, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0)
val_loader_a = DataLoader(ds_val_nih, **loader_args)
val_loader_b = DataLoader(ds_val_tb, **loader_args)
val_loader_c = DataLoader(ds_val_covid, **loader_args)
print(f"[DATA] train={len(train_loader):,} batches; validation NIH/TB/COVIDx={len(val_loader_a):,}/{len(val_loader_b):,}/{len(val_loader_c):,} batches")
print("[DATA] Datasets strict: ảnh thiếu/lỗi sẽ dừng chạy; không tạo black mock samples.")

if len(train_loader) < 2:
    raise RuntimeError(f"Unified train loader chỉ có {len(train_loader)} batch; cần ít nhất 2.")
for batch_index, smoke_batch in enumerate(train_loader):
    images = smoke_batch["images"]
    task_ids = smoke_batch["task_ids"]
    if images.ndim != 4 or tuple(images.shape[1:]) != (3, 512, 512):
        raise RuntimeError(f"Batch {batch_index}: image tensor shape không hợp lệ: {tuple(images.shape)}")
    if not torch.isfinite(images).all():
        raise RuntimeError(f"Batch {batch_index}: ảnh có NaN/Inf.")
    if set(task_ids.tolist()) != {0, 1, 2}:
        raise RuntimeError(f"Batch {batch_index}: thiếu task head; task_ids={sorted(set(task_ids.tolist()))}")
    if len(smoke_batch["labels"]) != images.shape[0]:
        raise RuntimeError(f"Batch {batch_index}: labels/images batch size không khớp.")
    if batch_index == 1:
        break
# The readiness probe consumed sampler RNG; restore the configured seed so the
# actual training epoch still starts from the deterministic first batch.
batch_sampler.rng = np.random.default_rng(batch_sampler.seed)
print("[PASS] Đã đọc và kiểm tra 2 mini-batch ảnh thật, đủ ba task, shape 3x512x512.")
print(">>> LUNGRAY KAGGLE PIPELINE IS VERIFIED & READY FOR SOTA TRAINING <<<")


In [ ]:
# Cell 5: Khởi tạo mô hình và nạp checkpoint baseline tốt nhất từ Notebook 02
from src.models.multi_task_net import MultiTaskChestXrayNet

# Best checkpoint from Notebook 02 (epoch selected by validation score).
# Kaggle Model mount provided for this run; override if the mounted version changes.
BASELINE_MODEL_DIR = os.environ.get(
    "LUNGRAY_BASELINE_MODEL_DIR",
    "/kaggle/input/models/phanphat/baseline-lungray/pytorch/default/1/checkpoint_baseline_multitask.pth",
).strip()
BASELINE_CHECKPOINT_PATH = os.environ.get("LUNGRAY_BASELINE_CHECKPOINT", "").strip()
checkpoint_candidates = [BASELINE_CHECKPOINT_PATH] if BASELINE_CHECKPOINT_PATH else []
checkpoint_candidates.extend([
    os.path.join(dataset_paths["output_dir"], "checkpoint_baseline_multitask.pth"),
    os.path.join(project_root, "results", "baseline", "checkpoint_baseline_multitask.pth"),
    os.path.join(project_root, "checkpoints", "checkpoint_baseline_multitask.pth"),
])
if os.path.isdir("/kaggle/input"):
    checkpoint_candidates.extend(
        glob.glob("/kaggle/input/**/checkpoint_baseline_multitask.pth", recursive=True)
    )

# Kaggle Models may expose a differently named file under the versioned mount.
if os.path.isfile(BASELINE_MODEL_DIR):
    baseline_model_files = [BASELINE_MODEL_DIR]
elif os.path.isdir(BASELINE_MODEL_DIR):
    baseline_model_files = [
        path for pattern in ("*.pth", "*.pt", "*.ckpt")
        for path in glob.glob(os.path.join(BASELINE_MODEL_DIR, "**", pattern), recursive=True)
    ]
else:
    baseline_model_files = []
preferred_model_names = {
    "checkpoint_baseline_multitask.pth", "best_model.pth", "best.pth", "model.pth",
}
checkpoint_candidates.extend(
    path for path in baseline_model_files
    if os.path.basename(path).lower() in preferred_model_names
)
if not any(os.path.isfile(path) for path in checkpoint_candidates if path):
    non_last_model_files = [
        path for path in baseline_model_files
        if os.path.basename(path).lower() != "last_model.pth"
    ]
    if len(non_last_model_files) == 1:
        checkpoint_candidates.append(non_last_model_files[0])
checkpoint_path = next((p for p in checkpoint_candidates if p and os.path.isfile(p)), None)
if checkpoint_path is None:
    available_pth = (
        glob.glob("/kaggle/input/**/*.pth", recursive=True)
        if os.path.isdir("/kaggle/input") else []
    )
    available_pth.extend(baseline_model_files)
    available_pth = list(dict.fromkeys(available_pth))
    has_last_model = any(os.path.basename(p).lower() == "last_model.pth" for p in available_pth)
    if has_last_model:
        detail = "Đang thấy last_model.pth nhưng đó là epoch gần nhất, không chắc là best checkpoint."
    else:
        detail = "Không tìm thấy file .pth trong Kaggle Input."
    raise FileNotFoundError(
        "Thiếu best baseline checkpoint checkpoint_baseline_multitask.pth. " + detail + " "
        "Upload đúng file này thành Kaggle Dataset hoặc attach Notebook 02 Output vào Notebook 03. "
        f"Đã quét Kaggle Model directory: {BASELINE_MODEL_DIR}. "
        "Nếu file best có đường dẫn khác, đặt env var LUNGRAY_BASELINE_CHECKPOINT="
        "/kaggle/input/<dataset>/checkpoint_baseline_multitask.pth. "
        f"Đã thử: {checkpoint_candidates}; các file .pth thấy được: {available_pth[:20]}"
    )

model = MultiTaskChestXrayNet(
    backbone_name="densenet121", pretrained=False, img_size=512, dropout_rate=0.2,
    num_classes_a=14, num_classes_b=1, num_classes_c=1, head_norm="layernorm",
).to(device)
try:
    baseline_checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
except TypeError:
    baseline_checkpoint = torch.load(checkpoint_path, map_location="cpu")
state_dict = baseline_checkpoint.get("model_state_dict", baseline_checkpoint)
load_result = model.load_state_dict(state_dict, strict=True)
assert not load_result.missing_keys and not load_result.unexpected_keys
print(f"[BASELINE] Nạp strict=True thành công: {checkpoint_path}")
print(f"[ARCH] DenseNet-121 shared backbone, 3 heads; {sum(p.numel() for p in model.parameters()):,} parameters")


In [ ]:
# Cell 6: ASL / BCE / Focal và optimizer hai giai đoạn
from src.models.losses import MaskedMultiTaskLoss

criterion = MaskedMultiTaskLoss(
    weights={"head_a": 1.0, "head_b": 0.8, "head_c": 0.8},
    head_a_criterion="asl", asl_gamma_neg=4.0, asl_gamma_pos=1.0, asl_clip=0.05,
    head_c_criterion="focal", focal_gamma=2.0, focal_alpha=0.25,
).to(device)

# Giữ nguyên split và sampler 60:25:15. ASL là điểm thử nghiệm cần so với baseline BCE.
# Giai đoạn 1 chỉ huấn luyện classifier heads; BN backbone cũng được giữ ở eval.
for parameter in model.backbone.parameters():
    parameter.requires_grad = False
optimizer_stage1 = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=3e-4, weight_decay=1e-5,
)
scheduler_stage1 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_stage1, T_max=3, eta_min=1e-5,
)
print("[LOSS] NIH=ASL(gamma_neg=4, gamma_pos=1, clip=.05); TB=BCE; COVIDx=Focal(gamma=2, alpha=.25)")
print("[OPT] Stage 1: frozen backbone, head LR=3e-4. Stage 2: backbone LR=1e-5, heads LR=1e-4.")


In [ ]:
# Cell 7: Two-stage fine-tuning, differential LR, AMP và best-checkpoint reload
from src.models.trainer import MultiTaskTrainer

assert device.type == "cuda", "Huấn luyện đầy đủ chỉ chạy trên Kaggle GPU."
EPOCHS_STAGE1 = 3
EPOCHS_STAGE2 = 22
MAX_BATCHES_PER_EPOCH = 600  # Khớp giới hạn throughput thực nghiệm baseline; đặt None để chạy full epoch.
MAX_VAL_BATCHES_PER_HEAD = None  # Full, deterministic validation tránh bias do thứ tự CSV.

common_trainer_args = dict(
    criterion=criterion, device=device, mixed_precision=True, early_stopping_patience=5,
    metric_weights={"auc_a": 0.40, "auc_b": 0.30, "auc_c": 0.30},
    max_batches_per_epoch=MAX_BATCHES_PER_EPOCH,
    max_val_batches_per_head=MAX_VAL_BATCHES_PER_HEAD, log_interval=50,
)

def run_training_stage(stage_name, optimizer, scheduler, epochs, save_name):
    """Runs one independently checkpointed training stage and returns its trainer/history."""
    stage_trainer = MultiTaskTrainer(
        model=model, optimizer=optimizer, lr_scheduler=scheduler,
        checkpoint_dir=os.path.join(dataset_paths["output_dir"], stage_name),
        results_dir=os.path.join(dataset_paths["results_dir"], stage_name),
        **common_trainer_args,
    )
    stage_history = stage_trainer.fit(
        train_loader=train_loader, val_loader_a=val_loader_a,
        val_loader_b=val_loader_b, val_loader_c=val_loader_c,
        epochs=epochs, save_name=save_name,
    )
    best_path = os.path.join(dataset_paths["output_dir"], stage_name, save_name)
    assert os.path.isfile(best_path), f"{stage_name}: best checkpoint chưa được tạo."
    best_payload = torch.load(best_path, map_location=device, weights_only=False)
    model.load_state_dict(best_payload["model_state_dict"], strict=True)
    return stage_trainer, stage_history, best_path

print(f"[STAGE 1] Train heads only: {EPOCHS_STAGE1} epochs; {MAX_BATCHES_PER_EPOCH} batches/epoch")
trainer_stage1, history_stage1, checkpoint_stage1 = run_training_stage(
    "sota_stage1_heads", optimizer_stage1, scheduler_stage1,
    EPOCHS_STAGE1, "best_stage1_heads.pth",
)

# Mở backbone sau khi nạp best Stage 1; optimizer mới vì param groups đã thay đổi.
for parameter in model.backbone.parameters():
    parameter.requires_grad = True
optimizer_stage2 = torch.optim.AdamW(
    [
        {"params": model.backbone.parameters(), "lr": 1e-5, "name": "backbone"},
        {"params": list(model.feature_norm.parameters()) + list(model.head_a.parameters()) + list(model.head_b.parameters()) + list(model.head_c.parameters()) + list(model.dropout.parameters()), "lr": 1e-4, "name": "heads"},
    ],
    weight_decay=1e-5,
)
scheduler_stage2 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_stage2, T_max=EPOCHS_STAGE2, eta_min=1e-6,
)
print(f"[STAGE 2] Unfreeze all: backbone LR=1e-5, heads LR=1e-4; up to {EPOCHS_STAGE2} epochs")
trainer_stage2, history_stage2, checkpoint_stage2 = run_training_stage(
    "sota_stage2_finetune", optimizer_stage2, scheduler_stage2,
    EPOCHS_STAGE2, "best_stage2_finetune.pth",
)

# Calibration/evaluation phải dùng best Stage 2, không phải trọng số epoch cuối.
best_payload = torch.load(checkpoint_stage2, map_location=device, weights_only=False)
model.load_state_dict(best_payload["model_state_dict"], strict=True)
model.eval()
history_df = pd.concat(
    [history_stage1.assign(stage="heads"), history_stage2.assign(stage="finetune", epoch=history_stage2["epoch"] + EPOCHS_STAGE1)],
    ignore_index=True,
)
history_df.to_csv(os.path.join(dataset_paths["results_dir"], "sota_combined_training_history.csv"), index=False)
print(f"[BEST] Stage 2 checkpoint reloaded: {checkpoint_stage2}")
print(f"[METRIC] Best Stage 2 validation metrics: {best_payload.get('val_metrics')}")


In [ ]:
# Cell 8: Hiệu chuẩn Youden's J từ validation thật (không dùng test)
from src.utils.threshold_tuning import (
    NIH_PATHOLOGY_NAMES, find_best_threshold_youden,
    save_thresholds_json, tune_multilabel_thresholds,
)

@torch.inference_mode()
def collect_validation_predictions(loader, head_name):
    """Returns concatenated ground truth and sigmoid probabilities for one task."""
    model.eval()
    probabilities, targets = [], []
    for batch in loader:
        logits = model(batch["images"].to(device))[head_name]
        probabilities.append(torch.sigmoid(logits).float().cpu().numpy())
        targets.append(torch.stack(batch["labels"]).float().cpu().numpy())
    if not probabilities:
        raise RuntimeError(f"Validation loader rỗng cho {head_name}.")
    return np.concatenate(targets, axis=0), np.concatenate(probabilities, axis=0)

y_true_a_val, y_prob_a_val = collect_validation_predictions(val_loader_a, "head_a")
y_true_b_val, y_prob_b_val = collect_validation_predictions(val_loader_b, "head_b")
y_true_c_val, y_prob_c_val = collect_validation_predictions(val_loader_c, "head_c")
thresholds_a = tune_multilabel_thresholds(
    y_true_a_val, y_prob_a_val, class_names=NIH_PATHOLOGY_NAMES, method="youden",
)
threshold_b, youden_b = find_best_threshold_youden(y_true_b_val, y_prob_b_val)
threshold_c, youden_c = find_best_threshold_youden(y_true_c_val, y_prob_c_val)
optimal_thresholds = {
    "calibration_split": "validation",
    "method": "youden_j",
    "head_a_nih_14": thresholds_a,
    "head_b_tb": {"threshold": round(float(threshold_b), 4), "youden_j": round(float(youden_b), 4)},
    "head_c_covid": {"threshold": round(float(threshold_c), 4), "youden_j": round(float(youden_c), 4)},
}
threshold_output = os.path.join(project_root, "configs", "optimal_thresholds_sota.json")
os.makedirs(os.path.dirname(threshold_output), exist_ok=True)
save_thresholds_json(optimal_thresholds, threshold_output)
print(f"[CALIBRATION] Saved validation-only Youden thresholds: {threshold_output}")
print(f"[CALIBRATION] TB tau={threshold_b:.4f}, J={youden_b:.4f}; COVIDx tau={threshold_c:.4f}, J={youden_c:.4f}")


In [ ]:
# Cell 9: Trực Quan Hóa Đồ Thị Thực Nghiệm Sống & Bảng Phân Tích Hiệu Năng
import matplotlib.pyplot as plt
import seaborn as sns

print("[*] Vẽ biểu đồ đường cong học tập (Loss Curves & Validation AUC) trực tiếp từ bộ nhớ phiên làm việc...")
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].plot(history_df['epoch'], history_df['train_loss'], label='Total Loss (ASL+Focal)', lw=2.5, color='#1E88E5', marker='o')
if 'train_loss_a' in history_df.columns:
    axes[0].plot(history_df['epoch'], history_df['train_loss_a'], label='Head A (ASL)', ls='--', color='#43A047')
if 'train_loss_b' in history_df.columns:
    axes[0].plot(history_df['epoch'], history_df['train_loss_b'], label='Head B (TB)', ls='--', color='#FB8C00')
if 'train_loss_c' in history_df.columns:
    axes[0].plot(history_df['epoch'], history_df['train_loss_c'], label='Head C (COVID Focal)', ls='--', color='#E53935')

axes[0].set_title('LungRay SOTA ASL Training Loss', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss Value')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(history_df['epoch'], history_df['val_combined_score'], label='Combined Score S_val', lw=2.5, color='#8E24AA', marker='s')
axes[1].axhline(y=0.80, color='gray', linestyle='--', alpha=0.6, label='Gate Target (0.80)')
axes[1].set_title('Validation Performance Convergence', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('ROC-AUC Score')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()
print("[OK] Đồ thị thực nghiệm sống đã được sinh thành công từ bộ nhớ! Tuân thủ 100% Ràng buộc 7.")

